## Exploratory Data Analysis (EDA)

Initial inspection of the SECOM dataset to understand structure, missing data, and feature behaviour before preprocessing.

In [3]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 50)

### Load dataset
Check shape and inspect a few rows.

In [4]:
df = pd.read_csv("../data/raw/uci-secom.csv")

print("Shape:", df.shape)
df.head()

Shape: (1567, 592)


,Time,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,...,566,567,568,569,570,571,572,573,574,575,576,577,578,579,580,581,582,583,584,585,586,587,588,589,Pass/Fail
0,2008-07-19 11:55:00,3030.93,2564.00,2187.7333,1411.1265,1.3602,100.0,97.6133,0.1242,1.5005,0.0162,-0.0034,0.9455,202.4396,0.0,7.9558,414.8710,10.0433,0.9680,192.3963,12.5190,1.4026,-5419.00,2916.50,-4043.75,...,NaN,NaN,NaN,NaN,533.8500,2.1113,8.95,0.3157,3.0624,0.1026,1.6765,14.9509,NaN,NaN,NaN,NaN,0.5005,0.0118,0.0035,2.3630,NaN,NaN,NaN,NaN,-1
1,2008-07-19 12:32:00,3095.78,2465.14,2230.4222,1463.6606,0.8294,100.0,102.3433,0.1247,1.4966,-0.0005,-0.0148,0.9627,200.5470,0.0,10.1548,414.7347,9.2599,0.9701,191.2872,12.4608,1.3825,-5441.50,2604.25,-3498.75,...,NaN,NaN,NaN,NaN,535.0164,2.4335,5.92,0.2653,2.0111,0.0772,1.1065,10.9003,0.0096,0.0201,0.0060,208.2045,0.5019,0.0223,0.0055,4.4447,0.0096,0.0201,0.0060,208.2045,-1
2,2008-07-19 13:17:00,2932.61,2559.94,2186.4111,1698.0172,1.5102,100.0,95.4878,0.1241,1.4436,0.0041,0.0013,0.9615,202.0179,0.0,9.5157,416.7075,9.3144,0.9674,192.7035,12.5404,1.4123,-5447.75,2701.75,-4047.00,...,0.4122,0.2562,0.4119,68.8489,535.0245,2.0293,11.21,0.1882,4.0923,0.0640,2.0952,9.2721,0.0584,0.0484,0.0148,82.8602,0.4958,0.0157,0.0039,3.1745,0.0584,0.0484,0.0148,82.8602,1
3,2008-07-19 14:43:00,2988.72,2479.90,2199.0333,909.7926,1.3204,100.0,104.2367,0.1217,1.4882,-0.0124,-0.0033,0.9629,201.8482,0.0,9.6052,422.2894,9.6924,0.9687,192.1557,12.4782,1.4011,-5468.25,2648.25,-4515.00,...,3.5611,0.0670,2.7290,25.0363,530.5682,2.0253,9.33,0.1738,2.8971,0.0525,1.7585,8.5831,0.0202,0.0149,0.0044,73.8432,0.4990,0.0103,0.0025,2.0544,0.0202,0.0149,0.0044,73.8432,-1
4,2008-07-19 15:22:00,3032.24,2502.87,2233.3667,1326.5200,1.5334,100.0,100.3967,0.1235,1.5031,-0.0031,-0.0072,0.9569,201.9424,0.0,10.5661,420.5925,10.3387,0.9735,191.6037,12.4735,1.3888,-5476.25,2635.25,-3987.50,...,NaN,NaN,NaN,NaN,532.0155,2.0275,8.83,0.2224,3.1776,0.0706,1.6597,10.9698,NaN,NaN,NaN,NaN,0.4800,0.4766,0.1045,99.3032,0.0202,0.0149,0.0044,73.8432,-1


### Target variable
Convert pass/fail labels into binary (0 = pass, 1 = fail).

In [5]:
df = df.rename(columns={"Pass/Fail": "target"})
df["target"] = df["target"].map({-1: 0, 1: 1})

df["target"].head()

0    0
1    0
2    1
3    0
4    0
Name: target, dtype: int64

### Target distribution
Check class balance before modelling.

In [6]:
target_counts = df["target"].value_counts()
target_percent = (df["target"].value_counts(normalize=True) * 100).round(2)

pd.DataFrame({
    "count": target_counts,
    "percent": target_percent
})

,count,percent
target,,
0,1463,93.36
1,104,6.64


### Feature columns
Most columns are numerical sensor readings.

In [7]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
numeric_cols.remove("target")

print("Numeric feature count:", len(numeric_cols))

Numeric feature count: 590


### Missing values
Identify columns with high missing rates.

In [8]:
missing_counts = df[numeric_cols].isnull().sum()
missing_percent = (missing_counts / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    "missing_count": missing_counts,
    "missing_percent": missing_percent
}).sort_values(by="missing_percent", ascending=False)

missing_df.head(15)

,missing_count,missing_percent
292,1429,91.19
293,1429,91.19
158,1429,91.19
157,1429,91.19
492,1341,85.58
85,1341,85.58
358,1341,85.58
220,1341,85.58
244,1018,64.96
517,1018,64.96


### High-missing columns
Columns above 50% missing will likely be removed.

In [9]:
high_missing_cols = missing_df[missing_df["missing_percent"] > 50]

print("Columns with >50% missing:", len(high_missing_cols))
high_missing_cols.head()

Columns with >50% missing: 28


,missing_count,missing_percent
292,1429,91.19
293,1429,91.19
158,1429,91.19
157,1429,91.19
492,1341,85.58


### Summary statistics
Check scale and variation across features.

In [10]:
df[numeric_cols].describe().T.head(10)

,count,mean,std,min,25%,50%,75%,max
0,1561.0,3014.452896,73.621787,2743.2400,2966.2600,3011.4900,3056.6500,3356.3500
1,1560.0,2495.850231,80.407705,2158.7500,2452.2475,2499.4050,2538.8225,2846.4400
2,1553.0,2200.547318,29.513152,2060.6600,2181.0444,2201.0667,2218.0555,2315.2667
3,1553.0,1396.376627,441.691640,0.0000,1081.8758,1285.2144,1591.2235,3715.0417
4,1553.0,4.197013,56.355540,0.6815,1.0177,1.3168,1.5257,1114.5366
5,1553.0,100.000000,0.000000,100.0000,100.0000,100.0000,100.0000,100.0000
6,1553.0,101.112908,6.237214,82.1311,97.9200,101.5122,104.5867,129.2522
7,1558.0,0.121822,0.008961,0.0000,0.1211,0.1224,0.1238,0.1286
8,1565.0,1.462862,0.073897,1.1910,1.4112,1.4616,1.5169,1.6564
9,1565.0,-0.000841,0.015116,-0.0534,-0.0108,-0.0013,0.0084,0.0749


### Low variance features
Some features have very little variation across rows, so they may not be useful for modelling.

In [11]:
variance = df[numeric_cols].var()
low_variance_cols = variance[variance < 1e-5]

print("Low-variance columns:", len(low_variance_cols))
low_variance_cols.head()

Low-variance columns: 148


5     0.0
13    0.0
42    0.0
49    0.0
52    0.0
dtype: float64

### Duplicates
Check for repeated rows.

In [12]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


### Correlation (sample)
A small subset of features is used to check basic relationships, since the full dataset is too large to visualise.

In [13]:
sample_cols = numeric_cols[:10]
df[sample_cols].corr()

,0,1,2,3,4,5,6,7,8,9
0,1.000000,-0.145071,0.004775,-0.007655,-0.011047,NaN,0.002281,0.031510,-0.052731,0.009052
1,-0.145071,1.000000,0.005802,-0.007603,-0.001641,NaN,-0.025702,-0.012084,0.031321,0.024015
2,0.004775,0.005802,1.000000,0.298935,0.095891,NaN,-0.136225,-0.273970,0.023609,0.016291
3,-0.007655,-0.007603,0.298935,1.000000,-0.058483,NaN,-0.685835,0.138290,-0.103656,0.068998
4,-0.011047,-0.001641,0.095891,-0.058483,1.000000,NaN,-0.074368,-0.916410,-0.026035,0.054619
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,0.002281,-0.025702,-0.136225,-0.685835,-0.074368,NaN,1.000000,0.024450,0.088280,-0.049864
7,0.031510,-0.012084,-0.273970,0.138290,-0.916410,NaN,0.024450,1.000000,0.020145,-0.092413
8,-0.052731,0.031321,0.023609,-0.103656,-0.026035,NaN,0.088280,0.020145,1.000000,-0.152133
9,0.009052,0.024015,0.016291,0.068998,0.054619,NaN,-0.049864,-0.092413,-0.152133,1.000000
